In [0]:
import pyspark.sql.functions as F
import pandas
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import os
from marel.services import ServiceProvider
from marel.databricks import DatabricksProvider
from pyspark.ml.feature import StandardScaler
from pyspark.ml.functions import vector_to_array

from pyspark.ml.classification import MultilayerPerceptronClassifier
import mlflow

provider = DatabricksProvider()
table_service = provider.table_service

In [0]:

df_csv = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .option("quote", "\"") \
    .option("delimiter", ",") \
    .load("/Volumes/teams/sensorx/data-dump/deviceID_Serial_GenSerial_machineCode.csv")

# Device IDs that have at least one 800W serial number
devices_800w = (
    df_csv.filter(F.col("gentype") == "800W")
    .select("properties_deviceId")
    .distinct()
)

In [0]:
from pyspark.ml.feature import VectorAssembler

def df_read_data(days, normalize=True):

    # Read telemetry data
    df_telemetry = table_service.load_table_from_device_checkpoint(
        table_name = "prod_medallion.silver_machine.sensorx_telemetry",
        machine_name="sensorx",
        max_historical_days=days,
        old_format=True
    ).select("properties_deviceID",
            "properties_payloadTimeStamp",
            "payload_xrayController_filamentCurrent",
            "payload_xrayController_onTime",
            "payload_xrayController_temperature",
            "payload_xrayController_tubeCurrent",
            "payload_xrayController_tubeVoltage"
            )

    # Filter to 800w devices
    df_telemetry_800w = df_telemetry.join(
        F.broadcast(devices_800w), df_telemetry["properties_deviceID"] == devices_800w["properties_deviceId"], "inner"
    ).drop(devices_800w["properties_deviceId"])

    # Reading fault tables
    df_fault = (
        table_service.load_table_from_device_checkpoint(
            table_name = "prod_medallion.silver_machine.pluto_xraycontroller_fault_property",
            machine_name="sensorx",
            max_historical_days=days,
            old_format=True
        )
        .filter(F.col("payload_fault_faultName") == "faultRegulation")
        .select("properties_payloadTimeStamp",
                "properties_deviceID",
                "payload_fault_state")
    )

    # --- Union + forward-fill ---
    telem_only = [c for c in df_telemetry_800w.columns
                if c not in ("properties_payloadTimeStamp", "properties_deviceID")]

    telem_part = df_telemetry_800w.select(
        "properties_payloadTimeStamp", "properties_deviceID",
        *telem_only,
        F.lit(None).cast("boolean").alias("payload_fault_state"),
        F.lit(True).alias("_is_telem"),
    )

    fault_part = df_fault.select(
        "properties_payloadTimeStamp", "properties_deviceID",
        *[F.lit(None).cast(dict(df_telemetry_800w.dtypes)[c]).alias(c)
        for c in telem_only],
        "payload_fault_state",
        F.lit(False).alias("_is_telem"),
    )

    w = Window.partitionBy("properties_deviceID").orderBy("properties_payloadTimeStamp")

    df = (
        telem_part.unionByName(fault_part)
        .withColumn("payload_fault_state",
                    F.last("payload_fault_state", ignorenulls=True).over(w))
        .filter(F.col("_is_telem"))
        .drop("_is_telem")
    )

    df = df.withColumn(
        "payload_fault_state", F.coalesce(F.col("payload_fault_state"), F.lit(False))
    )

    # Adding delta seconds feature
    df = (
        df
        .withColumn("prev_ts", F.lag("properties_payloadTimeStamp").over(w))
        .withColumn(
            "delta_seconds",
            F.when(
                F.col("prev_ts").isNull() |
                (F.col("properties_payloadTimeStamp").cast("long") - F.col("prev_ts").cast("long") < 0),
                None
            ).otherwise(
                F.col("properties_payloadTimeStamp").cast("long") - F.col("prev_ts").cast("long")
            ).cast("double")
        )
        .drop("prev_ts")
    )

    # Feature engineering
    sensor_cols = [c for c in df.columns
                   if c not in {"properties_deviceID", "properties_payloadTimeStamp", "payload_fault_state"}]

    df = df.na.drop(subset=sensor_cols)

    # --- Device ID index: use training data's mapping ---
    train_meta = spark.read.table("teams.sensorx.train_data").schema["features"].metadata
    device_vals = train_meta["ml_attr"]["attrs"]["nominal"][0]["vals"]
    default_index = float(len(device_vals) - 1) / 2.0
    device_mapping = spark.createDataFrame(
        [(dev_id, float(idx)) for idx, dev_id in enumerate(device_vals)],
        ["properties_deviceID", "deviceId_index"]
    )
    df = df.join(F.broadcast(device_mapping), on="properties_deviceID", how="left")
    df = df.withColumn("deviceId_index", F.coalesce(F.col("deviceId_index"), F.lit(default_index)))

    # Lag features
    n_lags = 20
    w_lag = Window.partitionBy("properties_deviceID").orderBy("properties_payloadTimeStamp")

    lag_exprs = [F.lag(col_name, L).over(w_lag).alias(f"{col_name}{L}")
                 for L in range(1, n_lags + 1) for col_name in sensor_cols]
    df = df.select("*", *lag_exprs)

    lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

    df = df.na.drop(subset=lag_cols)

    # --- Deviation features: 24-hour time-based rolling average ---
    w_daily = (
        Window.partitionBy("properties_deviceID")
        .orderBy(F.col("properties_payloadTimeStamp").cast("long"))
        .rangeBetween(-86400, 0)
    )

    dev_sensors = [
        "payload_xrayController_filamentCurrent",
        "payload_xrayController_temperature",
        "payload_xrayController_tubeCurrent",
    ]
    dev_cols = []
    dev_exprs = []

    for col_name in dev_sensors:
        avg_col = f"{col_name}_avg_daily"
        dev_col = f"{col_name}_dev_daily"
        dev_cols.extend([avg_col, dev_col])
        dev_exprs.append(F.avg(col_name).over(w_daily).alias(avg_col))
        dev_exprs.append((F.col(col_name) - F.avg(col_name).over(w_daily)).alias(dev_col))

    df = df.select("*", *dev_exprs)

    # Assemble features (133 total: 6 sensor + 120 lags + 6 deviation + 1 deviceId_index)
    feature_input_cols = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
    assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features")
    df_features = assembler.transform(df)

    df_features = df_features.select("properties_payloadTimeStamp", "properties_deviceID", "features")

    # Return raw features if normalization not needed (e.g., for GBT)
    if not normalize:
        return df_features

    # --- Normalize using training data statistics (for MLP) ---
    train_data = spark.read.table("teams.sensorx.train_data").select("features")

    scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
    scaler_model = scaler.fit(train_data)

    df_features_norm = scaler_model.transform(df_features).drop("features").withColumnRenamed("features_scaled", "features")

    return df_features_norm

In [0]:
days = 30
df_sx_data = df_read_data(days)

In [0]:
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

# Load model and run predictions
model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/5")
predictions = model.transform(df_sx_data).cache()

# Materialize now so downstream cells are instant
row_count = predictions.count()
print(f"Predictions materialized: {row_count:,} rows")

## Output

In [0]:
# --- MLP 15-day Model Inference: Top 5 with SHAP explanations ---
# Self-contained: requires cells 1-3 (imports, CSV, function definition)

import shap
import numpy as np
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import StructType, StructField

# 1. Load normalized features and run MLP inference
df_sx_data = df_read_data(30, normalize=True)

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"
mlp_model = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/5")
mlp_predictions = mlp_model.transform(df_sx_data).cache()

print(f"MLP 15-day predictions materialized: {mlp_predictions.count():,} rows")

# 2. Reconstruct feature names (133 total)
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_onTime",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "delta_seconds",
]
n_lags = 20
lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]
dev_sensors = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
]
dev_cols = []
for col in dev_sensors:
    dev_cols.extend([f"{col}_avg_daily", f"{col}_dev_daily"])
feature_names = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
short_names = [n.replace("payload_xrayController_", "").replace("_avg_daily", "_avg").replace("_dev_daily", "_dev") for n in feature_names]

# 3. Get top 5 devices (last 24h, max prob per device)
w_rank = Window.partitionBy("properties_deviceID").orderBy(F.col("fault_probability").desc())

mlp_scored = mlp_predictions.select(
    "properties_payloadTimeStamp",
    "properties_deviceID",
    "features",
    F.round(vector_to_array("probability")[1], 4).alias("fault_probability"),
)

top_5_spark = (
    mlp_scored
    .filter(F.col("properties_payloadTimeStamp") >= F.current_timestamp() - F.expr("INTERVAL 24 HOURS"))
    .withColumn("rn", F.row_number().over(w_rank))
    .filter(F.col("rn") == 1)
    .drop("rn", "properties_payloadTimeStamp")
    .orderBy(F.col("fault_probability").desc())
    .limit(5)
)

# Collect top 5 to driver
top_5_pd = top_5_spark.select(
    "properties_deviceID",
    vector_to_array("features").alias("features_array"),
    "fault_probability"
).toPandas()

X_explain = np.array(top_5_pd["features_array"].tolist())
device_ids = top_5_pd["properties_deviceID"].values
probs = top_5_pd["fault_probability"].values

# 4. Background sample for SHAP (50 random rows)
background_pd = (
    mlp_predictions
    .select(vector_to_array("features").alias("features_array"))
    .sample(fraction=0.0001, seed=42)
    .limit(50)
    .toPandas()
)
X_background = np.array(background_pd["features_array"].tolist())

print(f"\nSHAP setup: explaining {len(X_explain)} devices against {len(X_background)} background samples")

# 5. Predict function wrapping Spark MLP model
schema = StructType([StructField("features", VectorUDT(), True)])

def predict_fn_mlp15(X):
    rows = [(Vectors.dense(x.tolist()),) for x in X]
    df = spark.createDataFrame(rows, schema=schema)
    preds = mlp_model.transform(df).select(
        vector_to_array("probability")[1].alias("p")
    ).toPandas()
    return preds["p"].values

# 6. Run SHAP KernelExplainer
explainer = shap.KernelExplainer(predict_fn_mlp15, X_background)
shap_values = explainer.shap_values(X_explain, nsamples=200)

# 7. Display top 5 with top 3 SHAP features each
print(f"\n{'='*70}")
print(f"MLP Top 5 Machines at Risk (15-day horizon) + SHAP Explanations")
print(f"{'='*70}")

results_rows = []
for i in range(len(X_explain)):
    sv = shap_values[i]
    top_3_idx = np.argsort(np.abs(sv))[-3:][::-1]
    
    print(f"\n#{i+1} Device: {device_ids[i]}")
    print(f"   Fault probability: {probs[i]:.4f}")
    print(f"   Top 3 contributing features:")
    
    feature_contribs = []
    for rank, idx in enumerate(top_3_idx):
        direction = "+" if sv[idx] > 0 else "-"
        print(f"     {rank+1}. {short_names[idx]}: SHAP={sv[idx]:+.4f} (value={X_explain[i, idx]:.2f})")
        feature_contribs.append(f"{short_names[idx]} ({direction}{abs(sv[idx]):.3f})")
    
    results_rows.append((
        device_ids[i], float(probs[i]),
        feature_contribs[0], feature_contribs[1], feature_contribs[2]
    ))

results_df_15d = spark.createDataFrame(
    results_rows,
    ["machine_id", "fault_probability", "top_feature_1", "top_feature_2", "top_feature_3"]
)

print(f"\n")
display(results_df_15d)

In [0]:
# --- GBT Model Inference: Top 5 with SHAP explanations ---
# Self-contained: requires cells 1-3 (imports, CSV, function definition)

import shap
import numpy as np
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import StructType, StructField

# 1. Load raw features and run GBT inference
df_sx_data_raw = df_read_data(30, normalize=False)

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"
gbt_model = mlflow.spark.load_model("models:/teams.sensorx.gbt_fault_prediction/1")
gbt_predictions = gbt_model.transform(df_sx_data_raw).cache()

print(f"GBT predictions materialized: {gbt_predictions.count():,} rows")

# 2. Reconstruct feature names (133 total)
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_onTime",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "delta_seconds",
]
n_lags = 20
lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]
dev_sensors = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
]
dev_cols = []
for col in dev_sensors:
    dev_cols.extend([f"{col}_avg_daily", f"{col}_dev_daily"])
feature_names = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]

# Shortened names for display
short_names = [n.replace("payload_xrayController_", "").replace("_avg_daily", "_avg").replace("_dev_daily", "_dev") for n in feature_names]

# 3. Get top 5 devices (last 24h, max prob per device)
w_rank = Window.partitionBy("properties_deviceID").orderBy(F.col("fault_probability").desc())

gbt_scored = gbt_predictions.select(
    "properties_payloadTimeStamp",
    "properties_deviceID",
    "features",
    F.round(vector_to_array("probability")[1], 4).alias("fault_probability"),
)

top_5_spark = (
    gbt_scored
    .filter(F.col("properties_payloadTimeStamp") >= F.current_timestamp() - F.expr("INTERVAL 24 HOURS"))
    .withColumn("rn", F.row_number().over(w_rank))
    .filter(F.col("rn") == 1)
    .drop("rn", "properties_payloadTimeStamp")
    .orderBy(F.col("fault_probability").desc())
    .limit(5)
)

# Collect top 5 to driver
top_5_pd = top_5_spark.select(
    "properties_deviceID",
    vector_to_array("features").alias("features_array"),
    "fault_probability"
).toPandas()

X_explain = np.array(top_5_pd["features_array"].tolist())
device_ids = top_5_pd["properties_deviceID"].values
probs = top_5_pd["fault_probability"].values

# 4. Background sample for SHAP (50 random rows)
background_pd = (
    gbt_predictions
    .select(vector_to_array("features").alias("features_array"))
    .sample(fraction=0.0001, seed=42)
    .limit(50)
    .toPandas()
)
X_background = np.array(background_pd["features_array"].tolist())

print(f"\nSHAP setup: explaining {len(X_explain)} devices against {len(X_background)} background samples")

# 5. Predict function wrapping Spark GBT model
schema = StructType([StructField("features", VectorUDT(), True)])

def predict_fn(X):
    rows = [(Vectors.dense(x.tolist()),) for x in X]
    df = spark.createDataFrame(rows, schema=schema)
    preds = gbt_model.transform(df).select(
        vector_to_array("probability")[1].alias("p")
    ).toPandas()
    return preds["p"].values

# 6. Run SHAP KernelExplainer
explainer = shap.KernelExplainer(predict_fn, X_background)
shap_values = explainer.shap_values(X_explain, nsamples=200)

# 7. Display top 5 with top 3 SHAP features each
print(f"\n{'='*70}")
print(f"GBT Top 5 Machines at Risk (15-day horizon) + SHAP Explanations")
print(f"{'='*70}")

results_rows = []
for i in range(len(X_explain)):
    sv = shap_values[i]
    top_3_idx = np.argsort(np.abs(sv))[-3:][::-1]
    
    print(f"\n#{i+1} Device: {device_ids[i]}")
    print(f"   Fault probability: {probs[i]:.4f}")
    print(f"   Top 3 contributing features:")
    
    feature_contribs = []
    for rank, idx in enumerate(top_3_idx):
        direction = "+" if sv[idx] > 0 else "-"
        print(f"     {rank+1}. {short_names[idx]}: SHAP={sv[idx]:+.4f} (value={X_explain[i, idx]:.2f})")
        feature_contribs.append(f"{short_names[idx]} ({direction}{abs(sv[idx]):.3f})")
    
    results_rows.append((
        device_ids[i], float(probs[i]),
        feature_contribs[0], feature_contribs[1], feature_contribs[2]
    ))

# Also display as a table
results_df = spark.createDataFrame(
    results_rows,
    ["machine_id", "fault_probability", "top_feature_1", "top_feature_2", "top_feature_3"]
)

print(f"\n")
display(results_df)

In [0]:
# --- MLP 7-day Model Inference: Top 5 with SHAP explanations ---
# Self-contained: requires cells 1-3 (imports, CSV, function definition)

import shap
import numpy as np
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import StructType, StructField

# 1. Read raw features and normalize with 7-day training data
df_sx_data_raw_7d = df_read_data(30, normalize=False)

train_data_7d = spark.read.table("teams.sensorx.train_data_7d").select("features")
scaler_7d = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model_7d = scaler_7d.fit(train_data_7d)

df_sx_data_7d = scaler_model_7d.transform(df_sx_data_raw_7d).drop("features").withColumnRenamed("features_scaled", "features")

# 2. Load 7-day MLP model and run inference
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"
mlp_model_7d = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction_7d/1")
mlp_predictions_7d = mlp_model_7d.transform(df_sx_data_7d).cache()

print(f"MLP 7-day predictions materialized: {mlp_predictions_7d.count():,} rows")

# 3. Reconstruct feature names (133 total)
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_onTime",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "delta_seconds",
]
n_lags = 20
lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]
dev_sensors = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
]
dev_cols = []
for col in dev_sensors:
    dev_cols.extend([f"{col}_avg_daily", f"{col}_dev_daily"])
feature_names = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
short_names = [n.replace("payload_xrayController_", "").replace("_avg_daily", "_avg").replace("_dev_daily", "_dev") for n in feature_names]

# 4. Get top 5 devices (last 24h, max prob per device)
w_rank = Window.partitionBy("properties_deviceID").orderBy(F.col("fault_probability").desc())

mlp_7d_scored = mlp_predictions_7d.select(
    "properties_payloadTimeStamp",
    "properties_deviceID",
    "features",
    F.round(vector_to_array("probability")[1], 4).alias("fault_probability"),
)

top_5_spark_7d = (
    mlp_7d_scored
    .filter(F.col("properties_payloadTimeStamp") >= F.current_timestamp() - F.expr("INTERVAL 24 HOURS"))
    .withColumn("rn", F.row_number().over(w_rank))
    .filter(F.col("rn") == 1)
    .drop("rn", "properties_payloadTimeStamp")
    .orderBy(F.col("fault_probability").desc())
    .limit(5)
)

# Collect top 5 to driver
top_5_pd_7d = top_5_spark_7d.select(
    "properties_deviceID",
    vector_to_array("features").alias("features_array"),
    "fault_probability"
).toPandas()

X_explain_7d = np.array(top_5_pd_7d["features_array"].tolist())
device_ids_7d = top_5_pd_7d["properties_deviceID"].values
probs_7d = top_5_pd_7d["fault_probability"].values

# 5. Background sample for SHAP (50 random rows)
background_pd_7d = (
    mlp_predictions_7d
    .select(vector_to_array("features").alias("features_array"))
    .sample(fraction=0.0001, seed=42)
    .limit(50)
    .toPandas()
)
X_background_7d = np.array(background_pd_7d["features_array"].tolist())

print(f"\nSHAP setup: explaining {len(X_explain_7d)} devices against {len(X_background_7d)} background samples")

# 6. Predict function wrapping Spark MLP model
schema = StructType([StructField("features", VectorUDT(), True)])

def predict_fn_mlp7d(X):
    rows = [(Vectors.dense(x.tolist()),) for x in X]
    df = spark.createDataFrame(rows, schema=schema)
    preds = mlp_model_7d.transform(df).select(
        vector_to_array("probability")[1].alias("p")
    ).toPandas()
    return preds["p"].values

# 7. Run SHAP KernelExplainer
explainer_7d = shap.KernelExplainer(predict_fn_mlp7d, X_background_7d)
shap_values_7d = explainer_7d.shap_values(X_explain_7d, nsamples=200)

# 8. Display top 5 with top 3 SHAP features each
print(f"\n{'='*70}")
print(f"MLP Top 5 Machines at Risk (7-day horizon) + SHAP Explanations")
print(f"{'='*70}")

results_rows_7d = []
for i in range(len(X_explain_7d)):
    sv = shap_values_7d[i]
    top_3_idx = np.argsort(np.abs(sv))[-3:][::-1]
    
    print(f"\n#{i+1} Device: {device_ids_7d[i]}")
    print(f"   Fault probability: {probs_7d[i]:.4f}")
    print(f"   Top 3 contributing features:")
    
    feature_contribs = []
    for rank, idx in enumerate(top_3_idx):
        direction = "+" if sv[idx] > 0 else "-"
        print(f"     {rank+1}. {short_names[idx]}: SHAP={sv[idx]:+.4f} (value={X_explain_7d[i, idx]:.2f})")
        feature_contribs.append(f"{short_names[idx]} ({direction}{abs(sv[idx]):.3f})")
    
    results_rows_7d.append((
        device_ids_7d[i], float(probs_7d[i]),
        feature_contribs[0], feature_contribs[1], feature_contribs[2]
    ))

results_df_7d = spark.createDataFrame(
    results_rows_7d,
    ["machine_id", "fault_probability", "top_feature_1", "top_feature_2", "top_feature_3"]
)

print(f"\n")
display(results_df_7d)

In [0]:
# --- MLP v4 (15-day) Model Inference: Top 5 with SHAP explanations ---
# Self-contained: requires cells 1-3 (imports, CSV, function definition)

import shap
import numpy as np
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.types import StructType, StructField

# 1. Load normalized features and run MLP v4 inference
df_sx_data_v4 = df_read_data(30, normalize=True)

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"
mlp_model_v4 = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/4")
mlp_predictions_v4 = mlp_model_v4.transform(df_sx_data_v4).cache()

print(f"MLP v4 (15-day) predictions materialized: {mlp_predictions_v4.count():,} rows")

# 2. Reconstruct feature names (133 total)
sensor_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_onTime",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "delta_seconds",
]
n_lags = 20
lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]
dev_sensors = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
]
dev_cols = []
for col in dev_sensors:
    dev_cols.extend([f"{col}_avg_daily", f"{col}_dev_daily"])
feature_names = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
short_names = [n.replace("payload_xrayController_", "").replace("_avg_daily", "_avg").replace("_dev_daily", "_dev") for n in feature_names]

# 3. Get top 5 devices (last 24h, max prob per device)
w_rank = Window.partitionBy("properties_deviceID").orderBy(F.col("fault_probability").desc())

mlp_v4_scored = mlp_predictions_v4.select(
    "properties_payloadTimeStamp",
    "properties_deviceID",
    "features",
    F.round(vector_to_array("probability")[1], 4).alias("fault_probability"),
)

top_5_spark_v4 = (
    mlp_v4_scored
    .filter(F.col("properties_payloadTimeStamp") >= F.current_timestamp() - F.expr("INTERVAL 24 HOURS"))
    .withColumn("rn", F.row_number().over(w_rank))
    .filter(F.col("rn") == 1)
    .drop("rn", "properties_payloadTimeStamp")
    .orderBy(F.col("fault_probability").desc())
    .limit(5)
)

# Collect top 5 to driver
top_5_pd_v4 = top_5_spark_v4.select(
    "properties_deviceID",
    vector_to_array("features").alias("features_array"),
    "fault_probability"
).toPandas()

X_explain_v4 = np.array(top_5_pd_v4["features_array"].tolist())
device_ids_v4 = top_5_pd_v4["properties_deviceID"].values
probs_v4 = top_5_pd_v4["fault_probability"].values

# 4. Background sample for SHAP (50 random rows)
background_pd_v4 = (
    mlp_predictions_v4
    .select(vector_to_array("features").alias("features_array"))
    .sample(fraction=0.0001, seed=42)
    .limit(50)
    .toPandas()
)
X_background_v4 = np.array(background_pd_v4["features_array"].tolist())

print(f"\nSHAP setup: explaining {len(X_explain_v4)} devices against {len(X_background_v4)} background samples")

# 5. Predict function wrapping Spark MLP v4 model
schema = StructType([StructField("features", VectorUDT(), True)])

def predict_fn_mlp_v4(X):
    rows = [(Vectors.dense(x.tolist()),) for x in X]
    df = spark.createDataFrame(rows, schema=schema)
    preds = mlp_model_v4.transform(df).select(
        vector_to_array("probability")[1].alias("p")
    ).toPandas()
    return preds["p"].values

# 6. Run SHAP KernelExplainer
explainer_v4 = shap.KernelExplainer(predict_fn_mlp_v4, X_background_v4)
shap_values_v4 = explainer_v4.shap_values(X_explain_v4, nsamples=200)

# 7. Display top 5 with top 3 SHAP features each
print(f"\n{'='*70}")
print(f"MLP v4 Top 5 Machines at Risk (15-day horizon) + SHAP Explanations")
print(f"{'='*70}")

results_rows_v4 = []
for i in range(len(X_explain_v4)):
    sv = shap_values_v4[i]
    top_3_idx = np.argsort(np.abs(sv))[-3:][::-1]
    
    print(f"\n#{i+1} Device: {device_ids_v4[i]}")
    print(f"   Fault probability: {probs_v4[i]:.4f}")
    print(f"   Top 3 contributing features:")
    
    feature_contribs = []
    for rank, idx in enumerate(top_3_idx):
        direction = "+" if sv[idx] > 0 else "-"
        print(f"     {rank+1}. {short_names[idx]}: SHAP={sv[idx]:+.4f} (value={X_explain_v4[i, idx]:.2f})")
        feature_contribs.append(f"{short_names[idx]} ({direction}{abs(sv[idx]):.3f})")
    
    results_rows_v4.append((
        device_ids_v4[i], float(probs_v4[i]),
        feature_contribs[0], feature_contribs[1], feature_contribs[2]
    ))

results_df_v4 = spark.createDataFrame(
    results_rows_v4,
    ["machine_id", "fault_probability", "top_feature_1", "top_feature_2", "top_feature_3"]
)

print(f"\n")
display(results_df_v4)

In [0]:
# --- Combined Model Comparison: Global + Individual SHAP ---
# Requires cells 7-10 to have been run in this session

import numpy as np
import pandas as pd

# ============================================================
# TABLE 1: Model-Level Summary (global feature importance)
# ============================================================

# GBT: use native featureImportances (stages[-1] since MLflow wraps in PipelineModel)
gbt_importances = gbt_model.stages[-1].featureImportances.toArray()
gbt_top3_idx = np.argsort(gbt_importances)[-3:][::-1]

# MLPs: use mean |SHAP| across the 5 explained devices as proxy for global importance
mlp15_mean_shap = np.mean(np.abs(shap_values), axis=0)
mlp15_top3_idx = np.argsort(mlp15_mean_shap)[-3:][::-1]

mlp7d_mean_shap = np.mean(np.abs(shap_values_7d), axis=0)
mlp7d_top3_idx = np.argsort(mlp7d_mean_shap)[-3:][::-1]

mlpv4_mean_shap = np.mean(np.abs(shap_values_v4), axis=0)
mlpv4_top3_idx = np.argsort(mlpv4_mean_shap)[-3:][::-1]

def format_importance(names, indices, values):
    return [f"{names[i]} ({values[i]:.3f})" for i in indices]

model_summary_rows = [
    (
        "MLP v5 (15-day)", 0.8993, 
        f"{min(probs):.4f} - {max(probs):.4f}",
        *format_importance(short_names, mlp15_top3_idx, mlp15_mean_shap),
        "No"
    ),
    (
        "GBT v1 (15-day)", 0.9250,
        f"{min(probs):.4f} - {max(probs):.4f}".replace(str(min(probs)), f"{top_5_pd['fault_probability'].min():.4f}").replace(str(max(probs)), f"{top_5_pd['fault_probability'].max():.4f}") if False else f"{probs[0]:.4f} - {probs[-1]:.4f}",
        *format_importance(short_names, gbt_top3_idx, gbt_importances),
        "No"
    ),
    (
        "MLP 7-day", None,
        f"{min(probs_7d):.4f} - {max(probs_7d):.4f}",
        *format_importance(short_names, mlp7d_top3_idx, mlp7d_mean_shap),
        "No"
    ),
    (
        "MLP v4 (15-day)", 0.9154,
        f"{min(probs_v4):.4f} - {max(probs_v4):.4f}",
        *format_importance(short_names, mlpv4_top3_idx, mlpv4_mean_shap),
        "Yes"
    ),
]

print("="*90)
print("TABLE 1: MODEL-LEVEL SUMMARY")
print("="*90)
print(f"{'Model':<20} {'Test AUC':<10} {'Prob Range':<16} {'Global Top 1':<28} {'Global Top 2':<28} {'Global Top 3':<28} {'deviceId?'}")
print("-"*90)

# Recompute properly
gbt_probs_range = top_5_pd["fault_probability"] if "top_5_pd" in dir() else None

model_data = [
    ("MLP v5 (15d)", 0.8993, probs, mlp15_top3_idx, mlp15_mean_shap, False),
    ("GBT v1 (15d)", 0.9250, 
     np.array([0.9584, 0.9582, 0.9479, 0.9479, 0.9438]),  # from results
     gbt_top3_idx, gbt_importances, False),
    ("MLP 7-day", None, probs_7d, mlp7d_top3_idx, mlp7d_mean_shap, False),
    ("MLP v4 (15d)", 0.9154, probs_v4, mlpv4_top3_idx, mlpv4_mean_shap, True),
]

for name, auc, p, top3, vals, uses_devid in model_data:
    auc_str = f"{auc:.4f}" if auc else "N/A"
    prob_range = f"{p.min():.3f}-{p.max():.3f}"
    t1 = f"{short_names[top3[0]]} ({vals[top3[0]]:.3f})"
    t2 = f"{short_names[top3[1]]} ({vals[top3[1]]:.3f})"
    t3 = f"{short_names[top3[2]]} ({vals[top3[2]]:.3f})"
    devid = "YES \u26a0\ufe0f" if uses_devid else "No"
    print(f"{name:<20} {auc_str:<10} {prob_range:<16} {t1:<28} {t2:<28} {t3:<28} {devid}")

# ============================================================
# TABLE 2: All Predictions Combined (Spark DataFrame for display)
# ============================================================

# Build combined device-level table
all_rows = []

# MLP v5 15-day
for i in range(len(X_explain)):
    sv = shap_values[i]
    top_3_idx = np.argsort(np.abs(sv))[-3:][::-1]
    contribs = [f"{short_names[idx]} ({'+' if sv[idx]>0 else '-'}{abs(sv[idx]):.3f})" for idx in top_3_idx]
    all_rows.append(("MLP v5 (15d)", i+1, device_ids[i][:8]+"...", float(probs[i]), contribs[0], contribs[1], contribs[2]))

# GBT (use results_df from cell 8 which has the pre-computed SHAP features)
gbt_rows = results_df.collect()
for idx, row in enumerate(gbt_rows):
    all_rows.append(("GBT v1 (15d)", idx+1, row.machine_id[:8]+"...", float(row.fault_probability), row.top_feature_1, row.top_feature_2, row.top_feature_3))

# MLP 7-day
for i in range(len(X_explain_7d)):
    sv = shap_values_7d[i]
    top_3_idx = np.argsort(np.abs(sv))[-3:][::-1]
    contribs = [f"{short_names[idx]} ({'+' if sv[idx]>0 else '-'}{abs(sv[idx]):.3f})" for idx in top_3_idx]
    all_rows.append(("MLP 7-day", i+1, device_ids_7d[i][:8]+"...", float(probs_7d[i]), contribs[0], contribs[1], contribs[2]))

# MLP v4
for i in range(len(X_explain_v4)):
    sv = shap_values_v4[i]
    top_3_idx = np.argsort(np.abs(sv))[-3:][::-1]
    contribs = [f"{short_names[idx]} ({'+' if sv[idx]>0 else '-'}{abs(sv[idx]):.3f})" for idx in top_3_idx]
    all_rows.append(("MLP v4 (15d)", i+1, device_ids_v4[i][:8]+"...", float(probs_v4[i]), contribs[0], contribs[1], contribs[2]))

comparison_df = spark.createDataFrame(
    all_rows,
    ["model", "rank", "device", "fault_probability", "shap_feature_1", "shap_feature_2", "shap_feature_3"]
)

print(f"\n\n{'='*90}")
print("TABLE 2: ALL PREDICTIONS \u2014 DEVICE LEVEL (Top 5 per model)")
print(f"{'='*90}")
display(comparison_df)

# ============================================================
# TABLE 3: Cross-model device overlap
# ============================================================
devices_mlp15 = set(device_ids.tolist())
devices_gbt = set([r.machine_id for r in gbt_rows])
devices_7d = set(device_ids_7d.tolist())
devices_v4 = set(device_ids_v4.tolist())

all_flagged = devices_mlp15 | devices_gbt | devices_7d | devices_v4

print(f"\n\n{'='*90}")
print("TABLE 3: CROSS-MODEL OVERLAP")
print(f"{'='*90}")
print(f"Unique devices flagged across all models: {len(all_flagged)}")
print(f"\nDevices appearing in multiple models' top 5:")

overlap_rows = []
for dev in all_flagged:
    models_flagging = []
    if dev in devices_mlp15: models_flagging.append("MLP v5")
    if dev in devices_gbt: models_flagging.append("GBT")
    if dev in devices_7d: models_flagging.append("MLP 7d")
    if dev in devices_v4: models_flagging.append("MLP v4")
    if len(models_flagging) >= 2:
        overlap_rows.append((dev[:8]+"...", len(models_flagging), ", ".join(models_flagging)))
        print(f"  {dev[:12]}...  flagged by {len(models_flagging)} models: {', '.join(models_flagging)}")

if not overlap_rows:
    print("  None \u2014 no device appears in multiple top-5 lists")
else:
    overlap_df = spark.createDataFrame(overlap_rows, ["device", "num_models", "flagged_by"])
    display(overlap_df)

In [0]:
# --- Export comparison results to Excel ---
import subprocess
subprocess.check_call(["pip", "install", "openpyxl", "-q"])

import pandas as pd
import shutil

output_path = "/Volumes/teams/sensorx/data-dump/model_comparison_shap.xlsx"
temp_path = "/tmp/model_comparison_shap.xlsx"

# Table 1: Model-level summary
table1_data = [
    ("MLP v5 (15d)", 0.8993, f"{probs.min():.4f}-{probs.max():.4f}",
     short_names[mlp15_top3_idx[0]], f"{mlp15_mean_shap[mlp15_top3_idx[0]]:.3f}",
     short_names[mlp15_top3_idx[1]], f"{mlp15_mean_shap[mlp15_top3_idx[1]]:.3f}",
     short_names[mlp15_top3_idx[2]], f"{mlp15_mean_shap[mlp15_top3_idx[2]]:.3f}",
     "No"),
    ("GBT v1 (15d)", 0.9250, "0.9438-0.9584",
     short_names[gbt_top3_idx[0]], f"{gbt_importances[gbt_top3_idx[0]]:.3f}",
     short_names[gbt_top3_idx[1]], f"{gbt_importances[gbt_top3_idx[1]]:.3f}",
     short_names[gbt_top3_idx[2]], f"{gbt_importances[gbt_top3_idx[2]]:.3f}",
     "No"),
    ("MLP 7-day", None, f"{probs_7d.min():.4f}-{probs_7d.max():.4f}",
     short_names[mlp7d_top3_idx[0]], f"{mlp7d_mean_shap[mlp7d_top3_idx[0]]:.3f}",
     short_names[mlp7d_top3_idx[1]], f"{mlp7d_mean_shap[mlp7d_top3_idx[1]]:.3f}",
     short_names[mlp7d_top3_idx[2]], f"{mlp7d_mean_shap[mlp7d_top3_idx[2]]:.3f}",
     "No"),
    ("MLP v4 (15d)", 0.9154, f"{probs_v4.min():.4f}-{probs_v4.max():.4f}",
     short_names[mlpv4_top3_idx[0]], f"{mlpv4_mean_shap[mlpv4_top3_idx[0]]:.3f}",
     short_names[mlpv4_top3_idx[1]], f"{mlpv4_mean_shap[mlpv4_top3_idx[1]]:.3f}",
     short_names[mlpv4_top3_idx[2]], f"{mlpv4_mean_shap[mlpv4_top3_idx[2]]:.3f}",
     "Yes"),
]

df_table1 = pd.DataFrame(table1_data, columns=[
    "Model", "Test_AUC", "Prob_Range",
    "Global_Top1_Feature", "Top1_Importance",
    "Global_Top2_Feature", "Top2_Importance",
    "Global_Top3_Feature", "Top3_Importance",
    "Uses_deviceId"
])

# Table 2: Device-level predictions (from comparison_df)
df_table2 = comparison_df.toPandas()

# Table 3: Cross-model overlap
df_table3 = overlap_df.toPandas()

# Write to local temp first (Volumes FUSE doesn't support seek operations for xlsx)
with pd.ExcelWriter(temp_path, engine="openpyxl") as writer:
    df_table1.to_excel(writer, sheet_name="Model Summary", index=False)
    df_table2.to_excel(writer, sheet_name="Device Predictions", index=False)
    df_table3.to_excel(writer, sheet_name="Cross-Model Overlap", index=False)

# Copy to Volumes
shutil.copy(temp_path, output_path)

print(f"Exported to: {output_path}")
print(f"  Sheet 1: Model Summary ({len(df_table1)} rows)")
print(f"  Sheet 2: Device Predictions ({len(df_table2)} rows)")
print(f"  Sheet 3: Cross-Model Overlap ({len(df_table3)} rows)")